# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zainabaon/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [7]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/zainabaon/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    os.chdir("/content")
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

print("Working dir:", os.getcwd())

Working dir: /content/flyrank-ml-internship


## 1. Question

*The research question and the decision it supports.*

Which pages should be reviewed first for content refresh, given limited reviewer capacity? This supports a review-prioritization decision, not an automated content-editing decision. Lane: Refresh / Content Opportunity Scoring (Lane 2).

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

FlyRank internship warehouse release (build flyrank_pseudonymized_warehouse_release_v20260703): fact_content_daily_performance (month=2026-03, mid-panel month), dim_content, dim_clients — plus the anonymized starter dataset (30,000 rows) used for baseline/model iteration. Excluded: FlyRank product decision fields (health_score, priority_score, action_type) — not shipped, never used as features. No client names, domains, URLs, or raw queries used anywhere — only pseudonymized hash IDs for grouping.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

Task: scoring/ranking, built on a binary classifier. Label/proxy: is_declining_label = (trend_direction == "down") — a same-window proxy, disclosed as a limitation. Features: content_age_days, days_since_last_update, impressions_90d, avg_position, ctr, word_count — all observed, pre-decision signals; trend_pct deliberately excluded (leakage risk). Baseline: stale (180+ days) AND visible (500+ impressions) hand-written rule. Validation: client-grouped holdout split (75/25, zero client overlap, confirmed directly), also stress-tested against a naive random split to check for leakage (Section w06).

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [8]:
import pandas as pd
results = pd.DataFrame([
    {"method": "Baseline rule", "auc": None, "p20": 0.500, "p50": 0.560},
    {"method": "Logistic Regression", "auc": 0.540, "p20": 0.650, "p50": 0.660},
    {"method": "Decision Tree", "auc": 0.564, "p20": 0.500, "p50": 0.520},
    {"method": "Random Forest", "auc": 0.597, "p20": 0.550, "p50": 0.560},
])
print(results.to_string(index=False))

             method   auc  p20  p50
      Baseline rule   NaN 0.50 0.56
Logistic Regression 0.540 0.65 0.66
      Decision Tree 0.564 0.50 0.52
      Random Forest 0.597 0.55 0.56


Logistic Regression beat the baseline on Precision@50 (0.660 vs 0.560); Random Forest had the highest AUC but did not beat the baseline on this metric — added complexity did not automatically help. Re-validated under an honest client-grouped split (before: naive AUC 0.603/P@50 0.640, client overlap=32; after: grouped AUC 0.540/P@50 0.660, zero overlap) — Precision@50 held up under the stricter split.

## 5. Limitations

*What this work cannot claim.*

Same-window proxy label, not a validated future outcome. No causal claim that refresh causes recovery. Results are directional on this sample/split, not guaranteed to hold on new clients or the full warehouse. No Google algorithm claims made.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [9]:
print("17 pages flagged review_for_refresh (stale_visible_page reason code).")
print("12 of 17 (71%) concentrated in one client — flagged as a scoring-formula limitation, not necessarily a real finding.")
print("Human review required before acting; auto-publishing/de-indexing/cross-client comparison explicitly listed as no-go.")

17 pages flagged review_for_refresh (stale_visible_page reason code).
12 of 17 (71%) concentrated in one client — flagged as a scoring-formula limitation, not necessarily a real finding.
Human review required before acting; auto-publishing/de-indexing/cross-client comparison explicitly listed as no-go.


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

All charts and tables live directly in the deployed paper (results table, receipt-style metrics box, recommendations table) — built from work/outputs/ exports in w07_action_playbook.ipynb. Full reproducibility trail: docs/index.html ← w07_action_playbook.ipynb ← w05_model.ipynb ← w04_baseline_score.ipynb ← w03_data_contract.ipynb.

## Demo outline (5 minutes)
1. **Question** (30s): Which pages should a content team review first, given limited capacity?
2. **Method** (1min): Built a hand-written baseline, then compared 3 models under an honest client-holdout split.
3. **One chart** (1.5min): Show the results table — baseline 0.560 vs Logistic Regression 0.660 on Precision@50.
4. **One honest result** (1min): Complexity didn't always win — Random Forest had the best AUC but didn't beat the baseline on the metric that matters for a review queue.
5. **One recommendation** (1min): 17 real pages flagged for review, with reason codes and a human-review checklist — this is decision support, not automation.

## Shareable cuts

**Social post:**
Built a content-refresh priority model on real FlyRank search data this summer — compared a hand-written rule against 3 ML models under an honest client-holdout split (no data leakage). A simple Logistic Regression beat a Random Forest on the metric that actually mattered for the use case. Full write-up + reproducible notebooks: [your paper URL]

**Employer-facing summary:**
I built and validated a content-refresh prioritization model on real, large-scale (79M-row) search performance data, comparing multiple ML approaches against a transparent baseline under leakage-checked, client-grouped validation. The result is a decision-support tool — a ranked, reason-coded action queue — deployed as a public research paper with full reproducibility.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
